# vLLM-Hook Platform Parity Experiments

This notebook runs the same experiment suite as `tests/vllm_hook_experiments.py`, detects whether the active runtime should use `metal` or `non-metal`, detects the non-Metal runtime device as `cuda` or `cpu`, and logs metrics, records, manifests, and hook artifacts to Weights & Biases.

Use the same `BENCHMARK_PREFIX` or exact benchmark IDs on both machines to compare the two backends in W&B groups.

Runtime notes:
- On Colab, a GPU runtime (`Runtime > Change runtime type > GPU`) is recommended for the default 7B/8B models. CPU-only Colab is allowed but should use smaller models or very short smoke runs.
- On Apple Silicon with `vllm_metal` installed, this notebook selects `metal` automatically.
- `core-reranker` uses `mistralai/Mistral-7B-Instruct-v0.3` by default.
- `attn-tracker` uses `ibm-granite/granite-3.1-8b-instruct` by default.
- `steer-activation` uses `microsoft/Phi-3-mini-4k-instruct` by default.

## 1. Bootstrap Repo, Dependencies, And Backend

Run this first. On Colab it clones or refreshes `IBM/vLLM-Hook`, installs dependencies every fresh runtime, installs the local plugin package editable, detects whether the experiment backend should be `metal` or `non-metal`, and detects whether non-Metal should run on `cuda` or `cpu`.

Set `VLLM_HOOK_BACKEND=metal` or `VLLM_HOOK_BACKEND=non-metal` before running this cell to override detection. On Colab, `REPO_URL` / `REPO_BRANCH` must point at a branch or fork that contains `tests/vllm_hook_experiments.py`.


In [1]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import importlib.util
import platform

# On Colab this must point at a branch/fork that includes tests/vllm_hook_experiments.py.
REPO_URL = os.environ.get("VLLM_HOOK_REPO_URL", "https://github.com/tburleyinfo/vLLM-Hook.git")
REPO_BRANCH = os.environ.get("VLLM_HOOK_REPO_BRANCH", "vllm-hook-mlx")
REPO_NAME = "vLLM-Hook"

IN_COLAB = importlib.util.find_spec("google.colab") is not None
NOTEBOOK_DIR = Path.cwd()


def _repo_remote_matches(repo_root: Path, expected_remote: str) -> bool:
    """Return True when repo_root points at the expected git remote."""
    try:
        origin_url = subprocess.run(
            ["git", "-C", str(repo_root), "remote", "get-url", "origin"],
            check=True,
            capture_output=True,
            text=True,
        ).stdout.strip().removesuffix(".git")
    except Exception:
        return False
    return origin_url == expected_remote


def _find_existing_repo_root(start_dir: Path, expected_remote: str):
    """Walk upward from start_dir and return a matching repo root when one exists."""
    for candidate in [start_dir, *start_dir.parents]:
        if (candidate / ".git").exists() and _repo_remote_matches(candidate, expected_remote):
            return candidate
    return None


def _find_local_repo_root(start_dir: Path):
    """Walk upward from start_dir and return a vLLM-Hook checkout when one exists."""
    for candidate in [start_dir, *start_dir.parents]:
        if (candidate / "tests" / "vllm_hook_experiments.py").exists():
            return candidate
    for candidate in [Path.cwd(), Path.cwd() / REPO_NAME, Path.cwd().parent / REPO_NAME]:
        if (candidate / "tests" / "vllm_hook_experiments.py").exists():
            return candidate
    return None


if IN_COLAB:
    expected_remote = REPO_URL.removesuffix(".git")
    existing_repo_root = _find_existing_repo_root(NOTEBOOK_DIR, expected_remote)
    if existing_repo_root is not None:
        REPO_ROOT = existing_repo_root
        print(f"Colab detected. Reusing existing repo at {REPO_ROOT}")
    else:
        REPO_ROOT = Path("/content") / REPO_NAME
        if not REPO_ROOT.exists():
            print(f"Colab detected. Cloning {REPO_URL} ({REPO_BRANCH}) into {REPO_ROOT} ...")
            subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)], check=True)
        elif not _repo_remote_matches(REPO_ROOT, expected_remote):
            print(f"Remote mismatch under {REPO_ROOT}; replacing clone with {expected_remote}")
            shutil.rmtree(REPO_ROOT)
            subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)], check=True)
        else:
            print(f"Colab detected. Reusing existing clone at {REPO_ROOT}")
            print("Refreshing existing clone ...")
            subprocess.run(["git", "-C", str(REPO_ROOT), "fetch", "origin", REPO_BRANCH], check=True)
            subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", REPO_BRANCH], check=True)
            subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only", "origin", REPO_BRANCH], check=True)
    NOTEBOOK_DIR = REPO_ROOT / "notebooks"
    os.chdir(NOTEBOOK_DIR)
    print(f"Changed working directory to {NOTEBOOK_DIR}")
else:
    REPO_ROOT = _find_local_repo_root(NOTEBOOK_DIR)
    if REPO_ROOT is None:
        raise FileNotFoundError(
            "Could not find vLLM-Hook. Open this notebook from inside the repo "
            "or set REPO_ROOT manually."
        )

PROJECT_ROOT = REPO_ROOT
PKG_DIR = REPO_ROOT / "vllm_hook_plugins"
REQ_FILE = REPO_ROOT / "requirement.txt"
EXPERIMENT_RUNNER = REPO_ROOT / "tests" / "vllm_hook_experiments.py"

print("Colab       :", IN_COLAB)
print("Notebook dir:", NOTEBOOK_DIR)
print("Repo root   :", REPO_ROOT)
print("Repo branch :", REPO_BRANCH)
print("Package dir :", PKG_DIR)
print("Req file    :", REQ_FILE)
print("Runner file :", EXPERIMENT_RUNNER)

if not EXPERIMENT_RUNNER.exists():
    raise FileNotFoundError(
        f"Experiment runner not found: {EXPERIMENT_RUNNER}\n"
        "Colab cloned a repo/branch that does not contain the parity test runner. "
        "Push tests/vllm_hook_experiments.py to a branch or fork, then set "
        "VLLM_HOOK_REPO_URL and VLLM_HOOK_REPO_BRANCH before rerunning this cell."
    )

try:
    import torch
except Exception:
    torch = None
HAS_CUDA = bool(torch is not None and torch.cuda.is_available())
HAS_CUDART = importlib.util.find_spec("nvidia.cuda_runtime") is not None
if IN_COLAB and not HAS_CUDA and not HAS_CUDART:
    print(
        "Warning: no CUDA runtime detected. Non-Metal experiments will run on CPU; "
        "the default 7B/8B models may be slow or fail from memory pressure."
    )

if not PKG_DIR.exists():
    raise FileNotFoundError(f"Plugin directory not found: {PKG_DIR}")

if shutil.which("git") is None and IN_COLAB:
    raise RuntimeError("Colab was detected but git is unavailable in the runtime.")

# Colab runtimes are fresh; install every time. Local runtimes also run this so
# the notebook remains reproducible, but package managers will no-op cached deps.
if REQ_FILE.exists():
    req_cmd = [sys.executable, "-m", "pip", "install", "-r", str(REQ_FILE)]
    print("Running:", " ".join(req_cmd))
    subprocess.run(req_cmd, check=True)
else:
    print("Warning: requirement.txt not found; skipping dependency install.")

extra_cmd = [sys.executable, "-m", "pip", "install", "wandb", "pytest"]
print("Running:", " ".join(extra_cmd))
subprocess.run(extra_cmd, check=True)

protobuf_cmd = [sys.executable, "-m", "pip", "install", "--force-reinstall", "protobuf>=5.29.6,<6.30"]
print("Running:", " ".join(protobuf_cmd))
subprocess.run(protobuf_cmd, check=True)

editable_cmd = [sys.executable, "-m", "pip", "install", "-e", str(PKG_DIR)]
print("Running:", " ".join(editable_cmd))
subprocess.run(editable_cmd, check=True)

plugin_src_dir = str(PKG_DIR.resolve())
if plugin_src_dir not in sys.path:
    sys.path.insert(0, plugin_src_dir)
importlib.invalidate_caches()
try:
    import torch
except Exception:
    torch = None
HAS_CUDA = bool(torch is not None and torch.cuda.is_available())
HAS_CUDART = importlib.util.find_spec("nvidia.cuda_runtime") is not None


def detect_backend() -> str:
    override = os.environ.get("VLLM_HOOK_BACKEND")
    if override in {"metal", "non-metal"}:
        return override
    if platform.system() == "Darwin" and importlib.util.find_spec("vllm_metal") is not None:
        return "metal"
    return "non-metal"


def detect_runtime_device(backend: str) -> str:
    override = os.environ.get("VLLM_HOOK_DEVICE")
    if override in {"cuda", "cpu"}:
        return override
    if backend == "metal":
        return "metal"
    return "cuda" if HAS_CUDA else "cpu"


BACKEND = detect_backend()
RUNTIME_DEVICE = detect_runtime_device(BACKEND)
print("Plugin source:", plugin_src_dir)
print("Python exec  :", sys.executable)
print("Backend      :", BACKEND)
print("Device       :", RUNTIME_DEVICE)



Colab detected. Reusing existing clone at /content/vLLM-Hook
Refreshing existing clone ...
Changed working directory to /content/vLLM-Hook/notebooks
Colab       : True
Notebook dir: /content/vLLM-Hook/notebooks
Repo root   : /content/vLLM-Hook
Repo branch : vllm-hook-mlx
Package dir : /content/vLLM-Hook/vllm_hook_plugins
Req file    : /content/vLLM-Hook/requirement.txt
Runner file : /content/vLLM-Hook/tests/vllm_hook_experiments.py
Running: /usr/bin/python3 -m pip install -r /content/vLLM-Hook/requirement.txt
Running: /usr/bin/python3 -m pip install wandb pytest
Running: /usr/bin/python3 -m pip install --force-reinstall protobuf>=5.29.6,<6.30
Running: /usr/bin/python3 -m pip install -e /content/vLLM-Hook/vllm_hook_plugins
Plugin source: /content/vLLM-Hook/vllm_hook_plugins
Python exec  : /usr/bin/python3
Backend      : non-metal
Device       : cuda


## 3. Authenticate W&B And Hugging Face

For a temporary notebook workflow, paste keys into the constants below. If those are left blank, the cell falls back to environment variables, Colab Secrets, then an interactive prompt for the required W&B key. The Hugging Face token is optional unless the selected model requires it.

In [2]:
import getpass
import os

# Temporary local override. Leave blank to use env vars, Colab Secrets, or prompts.
HARDCODED_WANDB_API_KEY = ""
HARDCODED_HF_TOKEN = ""

try:
    from google.colab import userdata
except Exception:
    userdata = None

def secret_or_prompt(name: str, required: bool = False):
    hardcoded = {
        'WANDB_API_KEY': HARDCODED_WANDB_API_KEY,
        'HF_TOKEN': HARDCODED_HF_TOKEN,
    }.get(name)
    if hardcoded:
        os.environ[name] = hardcoded
        return hardcoded
    value = os.environ.get(name)
    if value:
        return value
    if userdata is not None:
        try:
            value = userdata.get(name)
        except Exception:
            value = None
        if value:
            os.environ[name] = value
            return value
    if required:
        value = getpass.getpass(f'{name}: ')
        os.environ[name] = value
        return value
    return None

secret_or_prompt('WANDB_API_KEY', required=True)
secret_or_prompt('HF_TOKEN', required=False)

if os.environ.get('HF_TOKEN'):
    os.environ['HUGGING_FACE_HUB_TOKEN'] = os.environ['HF_TOKEN']

import wandb
wandb.login(key=os.environ['WANDB_API_KEY'])

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: tm8ctgzqj8 (tm8ctgzqj8-georgia-institute-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## 4. Configure The W&B Comparison Group

Use the same group naming convention on both machines. Each experiment below uses `<BENCHMARK_PREFIX>-<experiment>` as the W&B group. For non-Metal runs, the prefix includes the detected runtime device (`cuda` or `cpu`).

In [3]:
WANDB_PROJECT = 'vllm-hook-platform-parity'
WANDB_ENTITY = 'tm8ctgzqj8-georgia-institute-of-technology'
BENCHMARK_PREFIX = f'{BACKEND}-{RUNTIME_DEVICE}-parity-001' if BACKEND == 'non-metal' else f'{BACKEND}-parity-001'
OUTPUT_DIR = PROJECT_ROOT / 'tests' / 'experiment_runs_colab'
GPU_MEMORY_UTILIZATION = '0.80'

print('W&B project:', WANDB_PROJECT)
print('backend:', BACKEND)
print('runtime device:', RUNTIME_DEVICE)
print('benchmark prefix:', BENCHMARK_PREFIX)
print('output dir:', OUTPUT_DIR)

W&B project: vllm-hook-platform-parity
backend: non-metal
runtime device: cuda
benchmark prefix: non-metal-cuda-parity-001
output dir: /content/vLLM-Hook/tests/experiment_runs_colab


## 5. Helper To Run An Experiment

This function shells out to `tests/vllm_hook_experiments.py` so the notebook path stays aligned with the checked-in test suite.

In [4]:
import subprocess
import sys
from datetime import datetime, timezone

def run_platform_experiment(
    experiment: str,
    *,
    benchmark_id: str | None = None,
    run_id: str | None = None,
    max_tokens: int | None = None,
    model: str | None = None,
    extra_args: list[str] | None = None,
):
    benchmark_id = benchmark_id or f'{BENCHMARK_PREFIX}-{experiment}'
    timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    run_id = run_id or f'{benchmark_id}-{BACKEND}-{timestamp}'
    cmd = [
        sys.executable,
        str(EXPERIMENT_RUNNER.relative_to(PROJECT_ROOT)),
        experiment,
        '--backend', BACKEND,
        '--benchmark-id', benchmark_id,
        '--run-id', run_id,
        '--output-dir', str(OUTPUT_DIR),
        '--gpu-memory-utilization', GPU_MEMORY_UTILIZATION,
        '--wandb-mode', 'online',
        '--wandb-project', WANDB_PROJECT,
        '--wandb-name-suffix', 'timestamp',
    ]
    if WANDB_ENTITY:
        cmd.extend(['--wandb-entity', WANDB_ENTITY])
    if BACKEND == 'non-metal':
        cmd.extend(['--device', RUNTIME_DEVICE])
    if max_tokens is not None:
        cmd.extend(['--max-tokens', str(max_tokens)])
    if model is not None:
        cmd.extend(['--model', model])
    if extra_args:
        cmd.extend(extra_args)

    print('Running:')
    print(' '.join(cmd))
    proc = subprocess.run(cmd, cwd=PROJECT_ROOT, text=True, capture_output=True)
    if proc.stdout:
        print(proc.stdout)
    if proc.stderr:
        print(proc.stderr)
    proc.check_returncode()

    local_dir = OUTPUT_DIR / 'platform_parity' / experiment / run_id
    print('Local output:', local_dir)
    return local_dir

## 6. Run One Experiment

Start with CoRe reranker, since that is the most direct comparison with your recent Metal run.

In [5]:
core_dir = run_platform_experiment('core-reranker')

Running:
/usr/bin/python3 tests/vllm_hook_experiments.py core-reranker --backend non-metal --benchmark-id non-metal-cuda-parity-001-core-reranker --run-id non-metal-cuda-parity-001-core-reranker-non-metal-20260512T220735Z --output-dir /content/vLLM-Hook/tests/experiment_runs_colab --gpu-memory-utilization 0.80 --wandb-mode online --wandb-project vllm-hook-platform-parity --wandb-name-suffix timestamp --wandb-entity tm8ctgzqj8-georgia-institute-of-technology --device cuda
Installed vLLM does not expose EngineArgs.device; recording requested device='cuda' without passing it to LLM.
INFO 05-12 22:07:47 [nixl_utils.py:20] Setting UCX_RCACHE_MAX_UNRELEASED to '1024' to avoid a rare memory leak in UCX when using NIXL.
WARNING 05-12 22:07:47 [nixl_utils.py:34] NIXL is not available
WARNING 05-12 22:07:47 [nixl_utils.py:44] NIXL agent config is not available
Initializing non-Metal vLLM-Hook backend model=mistralai/Mistral-7B-Instruct-v0.3 max_model_len=2048
INFO 05-12 22:07:47 [utils.py:233] n

Run attention tracker or activation steering when ready.

In [6]:
# attn_dir = run_platform_experiment('attn-tracker')
# steer_dir = run_platform_experiment('steer-activation', max_tokens=128)

Run all paired non-Metal experiments. This can take a while and requires enough GPU memory for the selected defaults.

In [7]:
# for exp in ['attn-tracker', 'core-reranker', 'steer-activation']:
#     kwargs = {'max_tokens': 128} if exp == 'steer-activation' else {}
#     run_platform_experiment(exp, **kwargs)

## 7. Inspect Local Outputs

The runner writes local JSON, CSV, manifest, and hook artifacts before uploading to W&B.

In [8]:
import json

def show_local_outputs(run_dir: Path):
    print('Files:')
    for path in sorted(run_dir.rglob('*')):
        if path.is_file():
            print(' ', path.relative_to(run_dir), path.stat().st_size, 'bytes')
    manifest_path = run_dir / 'artifact_manifest.json'
    if manifest_path.exists():
        manifest = json.loads(manifest_path.read_text())
        print('\nManifest files:', len(manifest['files']))
        for row in manifest['files']:
            if row['path'].endswith(('.pt', '.safetensors')):
                print(row['path'], row['bytes'], row['sha256'])

show_local_outputs(core_dir)

Files:
  artifact_manifest.json 1128 bytes
  core-reranker_non-metal.csv 159 bytes
  core-reranker_non-metal.json 2343 bytes
  hook_artifacts/4a61a423-f1e5-40c4-a7d5-52c6bea0c5c5/tp_rank_0/qk.pt 6045109 bytes
  hook_artifacts/4d4f20ea-c274-4c93-90ed-167f8d5f3ebe/tp_rank_0/qk.pt 515509 bytes
  hook_artifacts/RUN_ID.txt 74 bytes

Manifest files: 5
hook_artifacts/4a61a423-f1e5-40c4-a7d5-52c6bea0c5c5/tp_rank_0/qk.pt 6045109 6996121ba8ac000f452d7d4dfea67d8adf7a19af5b344b1b6f3585d4a95f7f22
hook_artifacts/4d4f20ea-c274-4c93-90ed-167f8d5f3ebe/tp_rank_0/qk.pt 515509 08af4768ddfdeed09acc4e0e876536545d7fd6b21203ce8116c118eb8b1b3fae


## 8. Retrieve The Latest Artifact From W&B

This downloads the latest logged artifact for an experiment/backend and lists Q/K cache files. Non-Metal Q/K caches are usually named `qk.pt`; Metal caches are usually `qkv.pt`.

In [11]:
from pathlib import Path
import wandb

def download_latest_artifact(experiment='core-reranker', backend=None):
    backend = backend or BACKEND
    api = wandb.Api()
    entity = WANDB_ENTITY or wandb.Api().default_entity
    runs = api.runs(f'{entity}/{WANDB_PROJECT}', order='-created_at', per_page=100)
    run = next(
        r for r in runs
        if r.name.startswith(f'{experiment}-{backend}')
        and experiment in r.tags
        and backend in r.tags
    )
    artifact = next(a for a in run.logged_artifacts() if a.type == 'vllm-hook-platform-parity')
    artifact_dir = Path(artifact.download())
    print('Run:', run.name, run.url)
    print('Artifact:', artifact.name)
    print('Downloaded to:', artifact_dir)
    for pattern in ['hook_artifacts/**/qk.pt', 'hook_artifacts/**/qkv.pt']:
        for path in sorted(artifact_dir.glob(pattern)):
            print(path)
    return artifact_dir

downloaded = download_latest_artifact('core-reranker')

wandb:   7 of 7 files downloaded.  


Run: core-reranker-non-metal-20260512T220929Z https://wandb.ai/tm8ctgzqj8-georgia-institute-of-technology/vllm-hook-platform-parity/runs/wo9o68jb
Artifact: core-reranker-non-metal-20260512T220929Z-non-metal-cuda-parity-001-core-reranker:v0
Downloaded to: /content/vLLM-Hook/notebooks/artifacts/core-reranker-non-metal-20260512T220929Z-non-metal-cuda-parity-001-core-reranker:v0
/content/vLLM-Hook/notebooks/artifacts/core-reranker-non-metal-20260512T220929Z-non-metal-cuda-parity-001-core-reranker:v0/hook_artifacts/4a61a423-f1e5-40c4-a7d5-52c6bea0c5c5/tp_rank_0/qk.pt
/content/vLLM-Hook/notebooks/artifacts/core-reranker-non-metal-20260512T220929Z-non-metal-cuda-parity-001-core-reranker:v0/hook_artifacts/4d4f20ea-c274-4c93-90ed-167f8d5f3ebe/tp_rank_0/qk.pt


## 9. Optional Tensor Summary

Use this after selecting one downloaded `qk.pt` or `qkv.pt` file.

In [12]:
import torch

def summarize_cache(path: str | Path):
    path = Path(path)
    cache = torch.load(path, map_location='cpu', weights_only=False)
    print('top-level keys:', list(cache.keys()))
    cache_key = 'qk_cache' if 'qk_cache' in cache else 'qkv_cache'
    print('cache key:', cache_key)
    for name, entry in cache[cache_key].items():
        print('\n', name)
        if isinstance(entry, dict):
            print('  keys:', list(entry.keys()))
            tokens = entry.get('tokens') or entry.get('q') or entry.get('k')
            if isinstance(tokens, list) and tokens:
                print('  tensors:', len(tokens), 'first shape:', tuple(tokens[0].shape), 'dtype:', tokens[0].dtype)
        else:
            print(type(entry))

summarize_cache('/content/vLLM-Hook/notebooks/artifacts/core-reranker-non-metal-20260512T220929Z-non-metal-cuda-parity-001-core-reranker:v0/hook_artifacts/4a61a423-f1e5-40c4-a7d5-52c6bea0c5c5/tp_rank_0/qk.pt')

top-level keys: ['config', 'qk_cache']
cache key: qk_cache

 model.layers.9.self_attn.attn
  keys: ['q', 'k_all', 'layer_num']
  tensors: 1 first shape: (118, 4096) dtype: torch.float16

 model.layers.12.self_attn.attn
  keys: ['q', 'k_all', 'layer_num']
  tensors: 1 first shape: (118, 4096) dtype: torch.float16

 model.layers.15.self_attn.attn
  keys: ['q', 'k_all', 'layer_num']
  tensors: 1 first shape: (118, 4096) dtype: torch.float16

 model.layers.16.self_attn.attn
  keys: ['q', 'k_all', 'layer_num']
  tensors: 1 first shape: (118, 4096) dtype: torch.float16

 model.layers.18.self_attn.attn
  keys: ['q', 'k_all', 'layer_num']
  tensors: 1 first shape: (118, 4096) dtype: torch.float16
